### Silver_Aircraft

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

df = spark.table("workspace.default.bronze_aircraft")

silver_aircraft = (
    df
    .dropDuplicates(["aircraft_id"])
    .dropna(subset=["aircraft_id"])
    .withColumn("capacity", F.col("capacity").cast(IntegerType()))
    .withColumn("manufacture_year", F.col("manufacture_year").cast(IntegerType()))
    .withColumn("processed_timestamp", F.current_timestamp())
)

silver_aircraft.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_aircraft")

print("✅ Processed Silver: aircraft")

### Silver_Flights

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

df = spark.table("workspace.default.bronze_flights")

silver_flights = (
    df
    .dropDuplicates(["flight_id"])
    .dropna(subset=["flight_id", "aircraft_id"])
    .withColumn("processed_timestamp", F.current_timestamp())
)

silver_flights.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_flights")

print("✅ Processed Silver: flights")

### Silver_Flights_incidents

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

df = spark.table("workspace.default.bronze_flight_incidents")

silver_flight_incidents = (
    df
    # Remove duplicate incidents
    .dropDuplicates(["incident_id"])

    # Remove records where important IDs are missing
    .dropna(subset=["incident_id", "flight_id", "aircraft_id"])

    # Convert incident timestamp to timestamp datatype
    .withColumn(
        "incident_timestamp",
        F.to_timestamp("incident_timestamp")
    )

    # Remove unwanted spaces from string columns
    .withColumn("incident_type", F.trim(F.col("incident_type")))
    .withColumn("severity", F.trim(F.col("severity")))
    .withColumn("resolution_status", F.trim(F.col("resolution_status")))
    .withColumn("airport", F.trim(F.col("airport")))

    # Add processing timestamp
    .withColumn(
        "processed_timestamp",
        F.current_timestamp()
    )
)

silver_flight_incidents.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_flight_incidents")

print("✅ Processed Silver: flight incidents")

### Silver_flights_sensor_data

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

df = spark.table("workspace.default.bronze_flight_sensor_data")

silver_sensor_data = (
    df
    # Remove duplicate sensor events
    .dropDuplicates(["sensor_id", "event_timestamp"])

    # Remove records with missing important IDs
    .dropna(subset=["sensor_id", "aircraft_id"])

    # Convert event timestamp to timestamp datatype
    .withColumn(
        "event_timestamp",
        F.to_timestamp("event_timestamp")
    )

    # Convert sensor value to numeric
    .withColumn(
        "sensor_value",
        F.col("sensor_value").cast(DoubleType())
    )

    # Remove unwanted spaces from string columns
    .withColumn(
        "sensor_id",
        F.trim(F.col("sensor_id"))
    )
    .withColumn(
        "aircraft_id",
        F.trim(F.col("aircraft_id"))
    )
    .withColumn(
        "sensor_type",
        F.trim(F.col("sensor_type"))
    )
    .withColumn(
        "unit",
        F.trim(F.col("unit"))
    )
    .withColumn(
        "anomaly_status",
        F.trim(F.col("anomaly_status"))
    )

    # Add processing timestamp
    .withColumn(
        "processed_timestamp",
        F.current_timestamp()
    )
)

silver_sensor_data.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_sensor_data")

print("✅ Processed Silver: sensor data")

### Silver_Maintenance

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

df = spark.table("workspace.default.bronze_maintenance")

silver_maintenance = (
    df
    # 1. Remove duplicate maintenance records
    .dropDuplicates(["maintenance_id"])

    # 2. Remove records with missing important IDs
    .dropna(subset=["maintenance_id", "aircraft_id"])

    # 3. Convert maintenance_date from string to date
    .withColumn(
        "maintenance_date",
        F.to_date("maintenance_date", "dd-MM-yyyy")
    )

    # 4. Convert downtime_hours to integer
    .withColumn(
        "downtime_hours",
        F.col("downtime_hours").cast(IntegerType())
    )

    # 5. Convert cost_usd to double
    .withColumn(
        "cost_usd",
        F.col("cost_usd").cast(DoubleType())
    )

    # 6. Remove unwanted spaces from string columns
    .withColumn(
        "maintenance_id",
        F.trim(F.col("maintenance_id"))
    )
    .withColumn(
        "aircraft_id",
        F.trim(F.col("aircraft_id"))
    )
    .withColumn(
        "maintenance_type",
        F.trim(F.col("maintenance_type"))
    )
    .withColumn(
        "severity",
        F.trim(F.col("severity"))
    )
    .withColumn(
        "status",
        F.trim(F.col("status"))
    )
    .withColumn(
        "technician",
        F.trim(F.col("technician"))
    )

    # 7. Add processing timestamp
    .withColumn(
        "processed_timestamp",
        F.current_timestamp()
    )
)

silver_maintenance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_maintenance")

print("✅ Processed Silver: maintenance")

### Silver_cockpit_display

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

df = spark.table("workspace.default.bronze_cockpit_display_config")

silver_display_unit = (
    df
    .dropDuplicates(["display_unit_id"])
    .dropna(subset=["display_unit_id"])
    
    .withColumn("display_unit_id", F.trim(F.col("display_unit_id")))
    .withColumn("display_type", F.trim(F.col("display_type")))
    .withColumn("screen_resolution", F.trim(F.col("screen_resolution")))
    .withColumn(
        "refresh_rate_hz",
        F.col("refresh_rate_hz").cast(IntegerType())
    )
    .withColumn("bus_connection", F.trim(F.col("bus_connection")))
    
    .withColumn("processed_timestamp", F.current_timestamp())
)

silver_display_unit.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_display_unit")

print("✅ Processed Silver: display unit")

### Silver_aerospace_flight_telemetry

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

df = spark.table("workspace.default.bronze_aerospace_flight_telemetry")

silver_flight_telemetry = (
    df
    # Remove duplicate telemetry records
    .dropDuplicates([
        "timestamp",
        "flight_id",
        "aircraft_tail_num",
        "flight_phase"
    ])

    # Required fields
    .dropna(subset=[
        "timestamp",
        "flight_id",
        "aircraft_tail_num"
    ])

    # Convert timestamp
    .withColumn(
        "timestamp",
        F.to_timestamp("timestamp", "dd-MM-yyyy HH:mm")
    )

    # Convert numeric columns
    .withColumn(
        "altitude_ft",
        F.col("altitude_ft").cast(DoubleType())
    )
    .withColumn(
        "indicated_airspeed_knots",
        F.col("indicated_airspeed_knots").cast(DoubleType())
    )
    .withColumn(
        "heading_deg",
        F.col("heading_deg").cast(DoubleType())
    )
    .withColumn(
        "pitch_deg",
        F.col("pitch_deg").cast(DoubleType())
    )
    .withColumn(
        "roll_deg",
        F.col("roll_deg").cast(DoubleType())
    )
    .withColumn(
        "pfd_voltage_v",
        F.col("pfd_voltage_v").cast(DoubleType())
    )
    .withColumn(
        "mfd_voltage_v",
        F.col("mfd_voltage_v").cast(DoubleType())
    )

    # Clean string columns
    .withColumn(
        "flight_id",
        F.trim(F.col("flight_id"))
    )
    .withColumn(
        "aircraft_tail_num",
        F.trim(F.col("aircraft_tail_num"))
    )
    .withColumn(
        "flight_phase",
        F.trim(F.col("flight_phase"))
    )
    .withColumn(
        "display_alert_flag",
        F.trim(F.col("display_alert_flag"))
    )

    # Processing timestamp
    .withColumn(
        "processed_timestamp",
        F.current_timestamp()
    )
)

silver_flight_telemetry.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_aerospace_flight_telemetry")

print("✅ Processed Silver: aerospace flight telemetry")

### Silver_avionics_flight_displays

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

df = spark.table("workspace.default.bronze_avionics_flight_displays")

silver_display_telemetry = (
    df
    .dropDuplicates([
        "timestamp",
        "flight_phase",
        "pfd_autoflight_mode",
        "mfd_active_page"
    ])
    .dropna(subset=[
        "timestamp",
        "flight_phase"
    ])
    .withColumn(
        "timestamp",
        F.to_timestamp("timestamp", "dd-MM-yyyy HH:mm")
    )
    .withColumn(
        "indicated_airspeed_kts",
        F.col("indicated_airspeed_kts").cast(DoubleType())
    )
    .withColumn(
        "altitude_ft",
        F.col("altitude_ft").cast(DoubleType())
    )
    .withColumn(
        "magnetic_heading_deg",
        F.col("magnetic_heading_deg").cast(DoubleType())
    )
    .withColumn(
        "pitch_attitude_deg",
        F.col("pitch_attitude_deg").cast(DoubleType())
    )
    .withColumn(
        "roll_attitude_deg",
        F.col("roll_attitude_deg").cast(DoubleType())
    )
    .withColumn(
        "pfd_bus_voltage_v",
        F.col("pfd_bus_voltage_v").cast(DoubleType())
    )
    .withColumn(
        "display_brightness_pct",
        F.col("display_brightness_pct").cast(DoubleType())
    )
    .withColumn(
        "flight_phase",
        F.trim(F.col("flight_phase"))
    )
    .withColumn(
        "pfd_autoflight_mode",
        F.trim(F.col("pfd_autoflight_mode"))
    )
    .withColumn(
        "mfd_active_page",
        F.trim(F.col("mfd_active_page"))
    )
    .withColumn(
        "processed_timestamp",
        F.current_timestamp()
    )
)

silver_display_telemetry.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.silver_display_telemetry"
    )

print("✅ Processed Silver: avionics flight displays")